In [1]:
!pip install dagshub mlflow optuna imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.

In [2]:
#!aws configure
#!aws configure
import dagshub
dagshub.init(repo_owner="rohitbedse",repo_name="yt-comment-sentiment-analysis",mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d05327bb-3341-45b0-a9eb-a44782945542&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5e8f56264043aa904fefce46c2b7ebd506877489b1cf66057a831ff3b9de07a5




Accessing as rohitbedse

Initialized MLflow to track repo "rohitbedse/yt-comment-sentiment-analysis"

Repository rohitbedse/yt-comment-sentiment-analysis initialized!

In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow")

In [4]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning With Chnaging Features")

<Experiment: artifact_location='mlflow-artifacts:/54b3eaefe0cb4e4d8246724361152697', creation_time=1775572221042, experiment_id='9', last_update_time=1775572221042, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning With Chnaging Features', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report,f1_score
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
import numpy as np

In [7]:
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [8]:
# -----------------------------
# STEP 1: Label Mapping
# -----------------------------
df = df.dropna(subset=['category'])
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# -----------------------------
# STEP 2: TRAIN-TEST SPLIT (FIRST)
# -----------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# -----------------------------
# STEP 3: TF-IDF (FIT ONLY ON TRAIN)
# -----------------------------
vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=5000)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# -----------------------------
# STEP 4: APPLY SMOTE ONLY ON TRAIN
# -----------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# -----------------------------
# STEP 5: MLflow logging function
# -----------------------------
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Bigram")
        mlflow.set_tag("experiment", "No_Leakage_Pipeline")

        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("resampling", "SMOTE")
        mlflow.log_param("vectorizer", "TF-IDF Bigram")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(model, f"{model_name}_model")


# -----------------------------
# STEP 6: OPTUNA OBJECTIVE
# -----------------------------
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )

    # ✅ SAME TRAIN/VAL SPLIT pattern
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_res, y_train_res,
        test_size=0.2,
        random_state=42,
        stratify=y_train_res
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    return f1_score(y_val, y_pred, average='macro')


# -----------------------------
# STEP 7: RUN OPTUNA
# -----------------------------
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_rf, n_trials=30)

    print("\n🏆 BEST PARAMS:", study.best_params)

    best_model = RandomForestClassifier(
        **study.best_params,
        random_state=42,
        n_jobs=-1
    )

    log_mlflow("RandomForest", best_model, X_train_res, X_test, y_train_res, y_test)


# -----------------------------
# RUN
# -----------------------------
run_optuna_experiment()

[I 2026-04-07 19:04:09,161] A new study created in memory with name: no-name-0ca402c3-3403-4685-bf4b-f0da8189c611
[I 2026-04-07 19:04:10,918] Trial 0 finished with value: 0.6737488100452996 and parameters: {'n_estimators': 101, 'max_depth': 17, 'min_samples_split': 9, 'min_samples_leaf': 20}. Best is trial 0 with value: 0.6737488100452996.
[I 2026-04-07 19:04:11,880] Trial 1 finished with value: 0.639087206752138 and parameters: {'n_estimators': 70, 'max_depth': 10, 'min_samples_split': 3, 'min_samples_leaf': 12}. Best is trial 0 with value: 0.6737488100452996.
[I 2026-04-07 19:04:14,030] Trial 2 finished with value: 0.6672775864850445 and parameters: {'n_estimators': 136, 'max_depth': 13, 'min_samples_split': 6, 'min_samples_leaf': 17}. Best is trial 0 with value: 0.6737488100452996.
[I 2026-04-07 19:04:22,901] Trial 3 finished with value: 0.7080773590125524 and parameters: {'n_estimators': 178, 'max_depth': 17, 'min_samples_split': 10, 'min_samples_leaf': 1}. Best is trial 3 with val


🏆 BEST PARAMS: {'n_estimators': 195, 'max_depth': 20, 'min_samples_split': 3, 'min_samples_leaf': 1}


2026/04/07 19:06:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 19:07:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_SMOTE_TFIDF_Bigram at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9/runs/6e909695bc47455c878bbd56321445f9
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9


In [9]:
# -----------------------------
# BEST PARAMS (RandomForest)
# -----------------------------
best_params = {
    'n_estimators': 195,
    'max_depth': 20,
    'min_samples_split': 3,
    'min_samples_leaf': 1
}

# -----------------------------
# MODEL INIT
# -----------------------------
model = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

# -----------------------------
# TRAIN ON SMOTE RESAMPLED DATA
# -----------------------------
model.fit(X_train_res, y_train_res)

# -----------------------------
# PREDICTIONS ON TEST SET
# -----------------------------
y_pred = model.predict(X_test)

# -----------------------------
# EVALUATION
# -----------------------------
print("Test Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Test Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

Test Macro F1: 0.6752950655831972
Test Weighted F1: 0.6957563703433323


In [10]:
import numpy as np

print("Unique predictions:", np.unique(y_pred))
print("Unique true labels:", np.unique(y_test))

Unique predictions: [0 1 2]
Unique true labels: [0 1 2]
